# 6.6 Project: deploy your capstone API

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langgraph==1.2.16"

## Your API key

Some cells in this notebook call a model and need your own OpenAI API key. Paste it when asked. It is stored only in this notebook session, never saved to the file.

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

**Prerequisites:** 4.2 (StateGraph, `START`/`END`, thread_id config), 4.4 (streaming modes), 5.1 (persistence and checkpointers), 6.1 (LangSmith tracing), 6.2 (evals), 6.3 (guardrails and budgets). Your capstone graph from the brief, working locally.

## Learning objectives

- Draw the service boundary: which parts of your capstone live in HTTP endpoints and which stay inside the graph.
- Build a FastAPI service around a LangGraph agent with a non-streaming `/chat` endpoint and an SSE `/chat/stream` endpoint.
- Replace `InMemorySaver` with `AsyncPostgresSaver` held for the whole app lifetime, and pass a real `thread_id` per conversation.
- Dockerize the service with a Postgres sidecar so the whole thing runs with one command.
- Ship a public demo safely: env-var secrets, rate limits, budgets, and tracing switched on.

## Why deployment is the project

Everything before this lesson ran on your laptop, where restarting the process was free and the only user was you. A deployed agent faces three things your laptop never did. First, strangers: requests arrive from people who did not read your README and will paste things you did not expect. Second, restarts: containers get recycled, and any state that lived in process memory evaporates with them. Third, bills: every model call now costs real money, charged to you, triggered by someone else.

This project is where the course earns its name. Module 2 gave you tools, Module 3 gave you RAG, Modules 4 and 5 gave you the graph, and 6.1 to 6.5 gave you tracing, evals, guardrails, and the EU angle. The capstone brief asks for a deployed demo because a reviewer can argue with a notebook, but they can *use* a URL. Employers consistently rank a live, evaluated, documented project above any number of tutorial clones: depth on one real system beats breadth on five.

The work comes in six steps. Do them in order; each step has a checkpoint you can verify before moving on.

## Step 1: the service boundary

The graph stays a graph. The HTTP layer does not contain agent logic; it translates between the web and the graph. Three jobs belong to HTTP: validate the incoming request (Pydantic, from 6.3), choose the thread (new or resumed), and shape the response. Everything else, retrieval, routing, tool calls, approvals, stays in LangGraph, where the evals from 6.2 already cover it.

In [ ]:
# service.py: step 1 skeleton (no streaming yet)
import os
import uuid
from contextlib import asynccontextmanager

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

from capstone_graph import build_graph  # your compiled-graph builder, from earlier modules

DATABASE_URL = os.environ["DATABASE_URL"]  # e.g. postgresql://user:pass@db:5432/agents


class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=4000)
    thread_id: str | None = None


@asynccontextmanager
async def lifespan(app: FastAPI):
    # The checkpointer is opened ONCE and lives as long as the app.
    from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver

    async with AsyncPostgresSaver.from_conn_string(DATABASE_URL) as checkpointer:
        await checkpointer.setup()  # creates checkpoint tables on first run
        app.state.graph = build_graph().compile(checkpointer=checkpointer)
        yield
    # connection closes here, on shutdown


app = FastAPI(lifespan=lifespan)


@app.post("/chat")
async def chat(req: ChatRequest):
    thread_id = req.thread_id or str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}
    try:
        result = await app.state.graph.ainvoke(
            {"messages": [("user", req.message)]}, config
        )
    except Exception as e:  # never leak tracebacks to the client
        raise HTTPException(status_code=500, detail="agent run failed") from e
    final = result["messages"][-1]
    return {"thread_id": thread_id, "reply": final.content}

Two things to notice. `from_conn_string` is an **async context manager** in 3.2.0, not a plain constructor: `AsyncPostgresSaver.from_conn_string(url)` without `async with` gives you a saver whose connection lifecycle is unmanaged. And the checkpointer is created in `lifespan`, not inside the endpoint: opening a database connection per request is slow and will exhaust the pool under load. Checkpoint for this step: `uvicorn service:app --reload` answers `POST /chat` with a `thread_id` you can reuse in the next call.

## Step 2: streaming with SSE

Nobody waits 40 seconds for a whole agent run to finish before seeing anything. Server-sent events let the frontend render tokens as they arrive. LangGraph's `stream_mode="messages"` yields `(chunk, metadata)` tuples as each model chunk lands, which is exactly what an SSE generator wants:

In [ ]:
# add to service.py
import json
from fastapi.responses import StreamingResponse


@app.post("/chat/stream")
async def chat_stream(req: ChatRequest):
    thread_id = req.thread_id or str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    async def event_generator():
        yield f"data: {json.dumps({'thread_id': thread_id})}\n\n"
        try:
            async for chunk, metadata in app.state.graph.astream(
                {"messages": [("user", req.message)]},
                config=config,
                stream_mode="messages",
            ):
                if chunk.content:  # skip empty metadata chunks
                    yield f"data: {json.dumps({'token': chunk.content})}\n\n"
        except Exception:
            yield f"data: {json.dumps({'error': 'agent run failed'})}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(event_generator(), media_type="text/event-stream")

Verify with curl, not the browser: `curl -N -X POST localhost:8000/chat/stream -H "Content-Type: application/json" -d '{"message":"summarise the refund policy"}'`. The `-N` flag matters: curl buffers output by default, which hides streaming and makes you think the endpoint is broken when it is not. The first event carries the `thread_id`; the frontend stores it and sends it back with the next message, which is how multi-turn conversation survives.

A note on the tuples: `stream_mode="messages"` yields pairs, not bare chunks. Code written for `"updates"` mode that unpacks one value will crash here. If you also want per-node progress (the "retrieving / reasoning / generating" phases from lesson 4.4), stream `["updates", "messages"]` and route each tuple by its mode, but keep the JSON event shape identical so the frontend has one parser.

## Step 3: durable state, for real this time

Lesson 5.1 taught persistence with `InMemorySaver`. That was a teaching tool. In production, two facts kill it: the process restarts, and there is more than one process. If your deployment runs two uvicorn workers behind a load balancer, a user whose first message hits worker A and whose second hits worker B gets two different conversations, because each worker holds its own memory. A shared Postgres checkpointer fixes both: every worker reads and writes the same checkpoint rows, keyed by `thread_id`.

The `lifespan` in step 1 already does this. Three production details on top:

1. **Table growth.** Every step of every run writes a checkpoint row. Under sustained traffic the tables grow without bound. Add a retention policy early: a scheduled job deleting checkpoints older than N days, or partitioning by time. This is the cheapest operational task on the list and the most commonly skipped.
2. **Thread ownership.** Whoever knows a `thread_id` can read that thread's history. If thread IDs are guessable (sequential integers) or the endpoint has no auth, one user reads another's conversation. Use unguessable UUIDs (step 1 does) and treat auth as a separate layer: even a shared demo password in front of the whole service is better than nothing, and it belongs in the HTTP layer, not the graph.
3. **Interrupts across restarts.** If your capstone uses human-in-the-loop approvals (lesson 5.2), the interrupt payload lives in the checkpoint, so a container restart mid-approval does not lose the pending run. Verify it: start a run that pauses for approval, restart the container, then resume with the same `thread_id`. If it resumes, your persistence is real.

## Step 4: Docker

The goal is one command that reproduces your whole system: `docker compose up`. The app needs Python, your pinned dependencies, and Postgres. Never bake secrets into the image; read them from the environment at runtime.

```dockerfile
# Dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
CMD ["uvicorn", "service:app", "--host", "0.0.0.0", "--port", "8000"]
```

```yaml
# docker-compose.yml
services:
  db:
    image: postgres:16
    environment:
      POSTGRES_USER: agents
      POSTGRES_PASSWORD: ${DB_PASSWORD}
      POSTGRES_DB: agents
    volumes:
      - pgdata:/var/lib/postgresql/data
  api:
    build: .
    depends_on:
      - db
    environment:
      DATABASE_URL: postgresql://agents:${DB_PASSWORD}@db:5432/agents
      OPENAI_API_KEY: ${OPENAI_API_KEY}
      LANGSMITH_TRACING: "true"
    ports:
      - "8000:8000"

volumes:
  pgdata:
```

`requirements.txt` pins the versions from the header of this lesson. Run `docker compose up --build`, then repeat the step 1 and step 2 checks against `localhost:8000`. If they pass inside compose, your system is reproducible: anyone with Docker gets the same stack, which is what "deployed demo" means in the capstone requirements.

## Step 5: a frontend, minimal on purpose

The frontend is not the project; it is a window onto the project. Ten lines of HTML with `EventSource` are enough, or skip the page entirely and document the curl commands. What matters is that a reviewer can interact without installing anything.

```html
<!-- demo.html: open in a browser, point at your deployed API -->
<script>
let threadId = null;
async function send() {
  const msg = document.getElementById("q").value;
  const body = JSON.stringify({message: msg, thread_id: threadId});
  const res = await fetch("/chat/stream", {
    method: "POST", headers: {"Content-Type": "application/json"}, body});
  const reader = res.body.getReader();
  const dec = new TextDecoder();
  let buf = "";
  while (true) {
    const {done, value} = await reader.read();
    if (done) break;
    buf += dec.decode(value);
    for (const line of buf.split("\n\n")) {
      if (line.startsWith("data: ")) {
        const evt = JSON.parse(line.slice(6));
        if (evt.thread_id) threadId = evt.thread_id;
        if (evt.token) document.getElementById("a").textContent += evt.token;
        if (evt.error) document.getElementById("a").textContent = evt.error;
      }
    }
    buf = buf.split("\n\n").pop();
  }
}
</script>
```

If your deployment sits behind nginx or a CDN, add the header `X-Accel-Buffering: no` on the stream response, or the proxy will buffer your SSE into one late blob and the "streaming" demo will look broken. Test the deployed URL with `curl -N` before you record the demo video; proxies are the number one reason a stream that worked locally fails in production.

## Step 6: shipping the demo

Pick the cheapest host that runs containers: a small VPS or a container platform with a free tier. The deployment checklist, in order:

1. **Secrets from the environment.** `OPENAI_API_KEY`, `DB_PASSWORD`, `LANGSMITH_API_KEY` as env vars or the platform's secret store. Never in the image, never in the repo.
2. **Tracing on.** `LANGSMITH_TRACING=true` from day one (lesson 6.1). Your first production incident will be a run that behaved oddly, and the trace is the only record of what happened.
3. **Budgets on.** The middleware from lesson 6.3, `ModelCallLimitMiddleware` and `ToolCallLimitMiddleware`, plus a rate limit on the HTTP endpoints. A public demo with no limits is an invitation for someone to run your agent in a loop on your bill.
4. **EU angle (lesson 6.5).** Host the database in the EU if your users are European, note it in the README, and make sure agent logs do not retain personal data longer than your stated policy.
5. **A status page of one line.** Log startup (checkpointer connected, graph compiled) so a failed deploy tells you which step died.

## The capstone requirements checklist

Before you call the project done, check it against the brief. Every row needs evidence, not a claim:

- **Custom tools:** at least two tools you wrote yourself, with Pydantic `args_schema` validation (2.2, 6.3).
- **RAG:** retrieval over a real document set with citations in the answer (3.1-3.4).
- **Multi-step LangGraph workflow:** three or more nodes with at least one conditional edge and one reducer (4.2, 4.3).
- **Eval set with results:** a golden dataset from 6.2, run against your agent, results committed to the repo with the date and model version.
- **Deployed demo:** a public URL serving the container from step 4, streaming working through the proxy.
- **README plus demo video:** problem statement, architecture, how to run it, eval results, costs, and what you would do next. The video shows the demo being used, not slides.

## What can go wrong

**The checkpointer is created per request.** Opening a Postgres connection inside the endpoint adds hundreds of milliseconds per call and exhausts the pool under concurrency. The saver belongs in `lifespan`, opened once. If your p99 latency climbs with traffic while the model calls stay flat, look here first.

**`from_conn_string` used as a constructor.** It is a context manager (async for the async saver). Calling it without `async with` leaves the connection unmanaged: it may work in a quick test and fail under load or on shutdown. The lifespan pattern in step 1 is the whole fix.

**InMemorySaver in a multi-worker deployment.** Two workers, two memories. The user experience is a conversation that randomly forgets. This fails silently: no errors, just amnesia. If your host runs more than one replica, the checkpointer must be shared (Postgres) and the store must be shared too (lesson 5.1's store discussion applies).

**SSE works locally, arrives as one blob in production.** A proxy or CDN buffered the stream. Add `X-Accel-Buffering: no`, test with `curl -N` against the deployed URL, and confirm the frontend parses incrementally. Do not "fix" this by making the agent faster; the transport was the problem.

**`stream_mode="messages"` unpacked wrong.** It yields `(chunk, metadata)` tuples. Unpacking one value raises `ValueError` at the first chunk, which in an SSE generator surfaces as a mysteriously empty stream. The tuple shape is documented in 4.4; when in doubt, print the first yielded item.

**Checkpoint tables grow forever.** Millions of rows, slowing every read, until the database is the bottleneck instead of the model. This is the write-storm problem practitioners warn about: every agent step writes state. Set a retention job before launch, not after the first slowdown.

**The demo bankrupts you.** A public endpoint with no rate limit and no call budgets is a loaded wallet on a public table. One visitor pasting a prompt that loops costs real money per iteration. Rate-limit the endpoints (per IP, per hour), keep the 6.3 middleware budgets tight for the demo tier, and set a billing alert on the model provider account.

**Secrets in the image.** An API key baked into a Docker layer is retrievable by anyone who pulls the image. Env vars or a secret store, reviewed with `docker history` before you push. If a key ever leaks, rotate it; do not just delete the image.

**Tracebacks to the client.** An unhandled exception in the endpoint returns a 500 with your stack trace, which leaks paths, library versions, and sometimes secrets in error context. Catch, log server-side, return a generic message. Step 1 shows the pattern.

## Check your understanding

**1. Why must the checkpointer be created in `lifespan` rather than inside the endpoint function?**
Opening a database connection per request is slow (hundreds of milliseconds of handshake) and leaks under concurrency: each request holds a connection until garbage collection, so traffic spikes exhaust the pool. `lifespan` opens one connection for the app's whole life, and every request shares it. The endpoint stays stateless, which is also what lets you run multiple workers.

**2. Your SSE stream works on localhost but the deployed frontend receives nothing until the run completes, then gets everything at once. Name two likely causes and how you confirm each.**
One: a reverse proxy or CDN is buffering the response. Confirm with `curl -N` against the deployed URL; if curl streams but the browser does not, it is the frontend parser, if curl also gets one blob, it is the proxy. Fix with `X-Accel-Buffering: no` on the stream response. Two: curl or the client was buffering. Confirm by adding `-N` to curl; if the stream appears, the endpoint was fine all along.

**3. You run two uvicorn workers with `InMemorySaver`. A user's first message hits worker A, the second hits worker B. Describe what the user experiences and the fix.**
The user experiences amnesia: worker B has no checkpoints for that `thread_id`, so the second message starts a fresh conversation. Nothing errors, which makes it hard to notice in testing with one worker. The fix is a shared checkpointer (Postgres, opened in `lifespan`), so every worker reads and writes the same rows keyed by `thread_id`.

**4. A visitor discovers your demo and triggers a run that makes 200 model calls against a 200k-token context. Which two controls from this course stop the damage, and where does each live?**
`ModelCallLimitMiddleware` (lesson 6.3) caps calls per run and lives in the agent's middleware stack; a per-IP rate limit on the HTTP endpoints lives in the FastAPI layer and caps runs per visitor. Belt and suspenders: the middleware bounds one run, the rate limit bounds one visitor. Neither replaces a billing alert on the provider account.

**5. In `stream_mode="messages"`, your generator does `async for chunk in graph.astream(..., stream_mode="messages")` and then `chunk.content`. It fails on the first iteration. Why, and what is the correct unpacking?**
The mode yields `(chunk, metadata)` tuples, not bare chunks, so `chunk` is actually the tuple and `.content` raises `AttributeError`. Correct: `async for chunk, metadata in graph.astream(..., stream_mode="messages")`, then guard with `if chunk.content`. The tuple shape is the single most common streaming bug in student projects.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.